# 3.3 LangChain Ecosystem Overview

## Playground Notebook

In this notebook, we'll explore the **LangChain ecosystem** hands-on using a **local Ollama model**.

| Topic | What You'll Learn |
|-------|-------------------|
| **What is LangChain** | The framework, its purpose, and the problem it solves |
| **The Five Pillars** | Core, LangChain, Community/Partners, LangGraph, LangSmith |
| **Core Building Blocks** | Models, Prompts, Chains, Output Parsers |
| **LCEL (Pipe Syntax)** | Composing components with the `\|` operator |
| **Memory** | Conversation history management |
| **Swappability** | How LangChain abstractions let you swap providers easily |

> **Model:** `llama3.2:3b` via **Ollama** — running 100% locally, no API keys needed.

---

In [1]:
# ============================================================
#  INSTALL DEPENDENCIES (run once)
# ============================================================
# !pip install langchain langchain-core langchain-ollama langchain-community
! pip install ipywidgets

In [11]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown

# ============================================================
#  CONFIGURATION
# ============================================================
MODEL = "llama3.2:3b" # "qwen2.5:1.5b" 

llm = ChatOllama(model=MODEL, temperature=0.7)

# Quick test — make sure Ollama is running
test = llm.invoke("Say 'hello' in one word.")
print(f"\u2705 Connected to Ollama | Model: {MODEL}")
print(f"   Test response: {test.content}")

✅ Connected to Ollama | Model: llama3.2:3b
   Test response: Hello.


---

## 1. Why LangChain? — The Problem It Solves

LLMs by themselves are **stateless text generators**. To build real applications you need:

- Prompt management
- Chaining multiple reasoning steps
- Connecting to external data
- Conversation memory
- Tool orchestration
- Error handling

**Without LangChain** — a Document Q&A system requires 1000+ lines of custom, tightly coupled code.

**With LangChain** — the same system can be built with ~30 lines of composable, modular code.

### The Five Pillars of the LangChain Ecosystem

| Pillar | Package | Purpose |
|--------|---------|----------|
| **1. Core** | `langchain-core` | Base abstractions, Runnable protocol, LCEL, message types |
| **2. LangChain** | `langchain` | Chains, agents, memory modules — cognitive architectures |
| **3. Community/Partners** | `langchain-ollama`, etc. | 700+ integrations — LLM providers, vector DBs, tools |
| **4. LangGraph** | `langgraph` | Stateful agents with graph-based control flow |
| **5. LangSmith** | `langsmith` | Observability — debugging, testing, monitoring |

Let's explore the most important ones hands-on.

---

## 2. Pillar 1 — LangChain Core: Message Types & the Runnable Protocol

Everything in LangChain implements the **Runnable protocol** — a universal interface with:

| Method | What It Does |
|--------|--------------|
| `invoke()` | Process a single input |
| `stream()` | Get output token by token |
| `batch()` | Process multiple inputs at once |

LangChain uses structured **message types** instead of raw dicts:

```
SystemMessage   →  Sets behavior, persona, constraints
HumanMessage    →  The user's input
AIMessage       →  The model's response
```

### Experiment 2A: Using Message Types Directly

In [12]:
# LangChain message types — structured, typed, and self-documenting
messages = [
    SystemMessage(content="You are a concise technical explainer. Max 2 sentences."),
    HumanMessage(content="What is LangChain?")
]

response = llm.invoke(messages)

print(f"Type: {type(response).__name__}")
print(f"Content: {response.content}")

Type: AIMessage
Content: LangChain is an open-source, Rust-based framework for building decentralized applications (dApps) and smart contracts on blockchain networks, particularly Ethereum, that integrates Web3 and blockchain technologies with machine learning (ML) and artificial intelligence (AI). It enables developers to create scalable, interoperable, and secure applications that leverage blockchain and ML/AI capabilities.


### Experiment 2B: The Runnable Protocol — `invoke()`, `stream()`, `batch()`

In [13]:
# Every LangChain component supports these three methods

# --- invoke: single input, full response ---
print("=" * 50)
print("invoke() — Single input, full response")
print("=" * 50)
result = llm.invoke("What is Python? One sentence.")
print(result.content)

invoke() — Single input, full response
Python is a high-level, interpreted programming language known for its simplicity, readability, and versatility, often used for web development, data analysis, artificial intelligence, and other applications.


In [14]:
# --- stream: token by token ---
print("=" * 50)
print("stream() — Tokens arrive as generated")
print("=" * 50)

for chunk in llm.stream("Name 3 programming languages. Brief."):
    print(chunk.content, end="", flush=True)
print()

stream() — Tokens arrive as generated
Here are 3 programming languages:

1. Java
2. Python
3. C++


In [15]:
# --- batch: multiple inputs at once ---
print("=" * 50)
print("batch() — Process multiple inputs")
print("=" * 50)

questions = [
    "What is an API? One sentence.",
    "What is a vector database? One sentence.",
    "What is RAG? One sentence."
]

results = llm.batch(questions)

for q, r in zip(questions, results):
    print(f"Q: {q}")
    print(f"A: {r.content}\n")

batch() — Process multiple inputs
Q: What is an API? One sentence.
A: An API, or Application Programming Interface, is a set of defined rules and protocols that enables different software systems to communicate with each other, sharing data and functionality in a standardized way.

Q: What is a vector database? One sentence.
A: A vector database is a type of data storage system that organizes data as vectors, allowing for efficient similarity searches, clustering, and retrieval of data points in high-dimensional spaces, often used in applications such as text search, image recognition, and recommender systems.

Q: What is RAG? One sentence.
A: RAG can refer to several things, but some common interpretations include:

- RAG (Radioactive Assay Guidance) - a system used in nuclear medicine for measuring the activity of radioactive substances.
- RAG (Resin-Assisted Gold) - a dental restoration technique that uses a resin to bond gold restorations to teeth.
- RAG (Ragtag) - a colloquial ter

---

## 3. Prompt Templates — Reusable, Parameterized Prompts

Instead of building prompt strings manually, LangChain provides **ChatPromptTemplate** — a reusable template with variables.

```python
# Manual string building (fragile, error-prone)
prompt = f"Explain {topic} to a {audience}"

# LangChain template (reusable, composable)
template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}."),
    ("human", "Explain {topic} to a {audience}.")
])
```

### Experiment 3A: Creating and Using Prompt Templates

In [16]:
# Define a reusable prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Keep answers to 2-3 sentences."),
    ("human", "Explain {topic} to a {audience}.")
])

# Format with specific values
messages = prompt.invoke({
    "role": "friendly teacher",
    "topic": "LangChain",
    "audience": "beginner programmer"
})

# See what was generated
print("Formatted messages:")
for msg in messages.messages:
    print(f"  [{type(msg).__name__}] {msg.content}")

print("\nResponse:")
response = llm.invoke(messages)
display(Markdown(response.content))

Formatted messages:
  [SystemMessage] You are a friendly teacher. Keep answers to 2-3 sentences.
  [HumanMessage] Explain LangChain to a beginner programmer.

Response:


LangChain is an open-source framework that helps developers build scalable, modular, and extensible blockchain applications. It provides a simple and Pythonic API for interacting with blockchain data, making it easier to integrate blockchain functionality into existing projects. Think of LangChain as a set of tools that help you "chain" together blockchain-related code to create powerful and flexible applications.

In [17]:
# Reuse the SAME template with different inputs
scenarios = [
    {"role": "scientist", "topic": "vector embeddings", "audience": "high school student"},
    {"role": "chef", "topic": "APIs", "audience": "someone who only knows cooking"},
]

for scenario in scenarios:
    print(f"\n{'=' * 50}")
    print(f"Role: {scenario['role']} | Topic: {scenario['topic']}")
    print(f"{'=' * 50}")
    messages = prompt.invoke(scenario)
    response = llm.invoke(messages)
    display(Markdown(response.content))


Role: scientist | Topic: vector embeddings


Vector embeddings are a way to represent words or objects as tiny, mathematical vectors that capture their meaning. Think of it like a map that shows how words are related to each other, allowing machines to understand the context and nuances of language. For example, the word "dog" and "pet" are represented as nearby points on this map, indicating their strong semantic connection.


Role: chef | Topic: APIs


Think of an API like a recipe book that allows different chefs (developers) to access and use each other's ingredients (data) to create new dishes (applications). Just as a chef would follow a recipe to create a dish, an API provides a set of instructions (protocol) for other applications to use its data, allowing them to create new and innovative recipes (services).

---

## 4. LCEL — LangChain Expression Language (The Pipe Operator)

LCEL is LangChain's **declarative composition syntax**. You chain components together using the pipe `|` operator — just like Unix pipes.

```
prompt | model | output_parser
```

Each component takes input from the previous one:

```
{variables} → PromptTemplate → Messages → ChatModel → AIMessage → StrOutputParser → string
```

The result is a **chain** — a new Runnable that supports `invoke()`, `stream()`, and `batch()`.

### Experiment 4A: Building Your First Chain with LCEL

In [18]:
# Three components snapped together with |
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("human", "{question}")
])

parser = StrOutputParser()  # Extracts just the text from AIMessage

# Build the chain
chain = prompt | llm | parser

print(f"Chain type: {type(chain).__name__}")
print(f"Chain components: PromptTemplate | ChatOllama | StrOutputParser")

# invoke() — just pass the variables
result = chain.invoke({"question": "What are the 5 pillars of LangChain?"})

print(f"\nResult type: {type(result).__name__}")  # str, not AIMessage!
display(Markdown(result))

Chain type: RunnableSequence
Chain components: PromptTemplate | ChatOllama | StrOutputParser

Result type: TextAccessor


LangChain is an open-source framework for building blockchain-agnostic applications. The 5 pillars of LangChain are:

1. **API**: A standardized API for interacting with blockchain-based services.
2. **Data Model**: A data model that enables the creation of custom data types and structures.
3. **Chain-agnostic**: LangChain applications can be built on various blockchain networks (e.g., Ethereum, Solana, Binance Smart Chain).
4. **Service-oriented**: LangChain applications are composed of modular, service-oriented components.
5. **Extensible**: LangChain is designed to be extensible through plugins and custom code.

### Experiment 4B: Streaming Through a Chain

Because every component implements the Runnable protocol, the entire chain supports streaming automatically.

In [19]:
# Streaming works through the entire chain
print("Streaming through the chain:\n")

for chunk in chain.stream({"question": "What is LCEL in LangChain? Brief explanation."}):
    print(chunk, end="", flush=True)

print()

Streaming through the chain:

In LangChain, LCEL is an alias for the `LinkChainEntity` class. It represents a link entity in the chain, containing metadata such as the entity's id, type, and title.


### Experiment 4C: A More Useful Chain — Topic Explainer

In [20]:
# A reusable chain with multiple template variables
explainer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {domain} instructor. Explain concepts clearly in {length}."),
    ("human", "Explain: {concept}")
])

explainer_chain = explainer_prompt | llm | StrOutputParser()

# Use it for different topics
topics = [
    {"domain": "AI/ML", "concept": "the Runnable protocol in LangChain", "length": "3 sentences"},
    {"domain": "software engineering", "concept": "why modular architecture matters", "length": "2 sentences"},
]

for topic in topics:
    print(f"\n{'=' * 50}")
    print(f"Concept: {topic['concept']}")
    print(f"{'=' * 50}")
    result = explainer_chain.invoke(topic)
    display(Markdown(result))


Concept: the Runnable protocol in LangChain


Here's an explanation of the Runnable protocol in LangChain:

The Runnable protocol in LangChain is a decentralized, modular, and open-source framework that enables the creation of self-executing, version-controlled, and interoperable AI models. It uses a unique architecture that allows models to run in a sandboxed environment, ensuring that each execution is isolated and tamper-proof, while also providing a flexible and scalable framework for model deployment and management. By leveraging the Runnable protocol, developers can build and deploy AI models that are secure, reliable, and easily updatable, without relying on a central authority or a specific cloud provider.


Concept: why modular architecture matters


Modular architecture matters because it allows for greater flexibility, scalability, and maintainability in software development, enabling developers to easily swap out or update individual components without affecting the entire system. By breaking down complex systems into smaller, independent modules, developers can also improve code reuse, reduce coupling, and increase overall efficiency and reliability.

---

## 5. Pillar 2 — Chains & Memory

LangChain provides **memory modules** to manage conversation history, since LLMs have no built-in memory.

| Memory Type | How It Works |
|-------------|--------------|
| **ConversationBufferMemory** | Stores the full conversation history |
| **ConversationBufferWindowMemory** | Keeps only the last K exchanges |
| **ConversationSummaryMemory** | LLM-generated summary of the conversation |

### Experiment 5A: Manual Conversation Memory

The simplest approach — manage the message list yourself.

In [21]:
# Manual conversation memory — you control the message list
conversation = [
    SystemMessage(content="You are a helpful AI tutor. Keep answers to 1-2 sentences.")
]

def chat_turn(user_input):
    """Add user message, get response, save to history."""
    conversation.append(HumanMessage(content=user_input))
    response = llm.invoke(conversation)
    conversation.append(response)  # AIMessage saved to history
    return response.content

# Multi-turn conversation
turns = [
    "What is LangChain?",
    "What problem does it solve?",
    "What was my first question?"
]

for i, user_msg in enumerate(turns, 1):
    print(f"\n{'=' * 50}")
    print(f"Turn {i}")
    print(f"{'=' * 50}")
    print(f"User: {user_msg}")
    answer = chat_turn(user_msg)
    print(f"AI:   {answer}")

print(f"\nTotal messages in history: {len(conversation)}")
print("The model remembers context because we pass the full history each time!")


Turn 1
User: What is LangChain?
AI:   LangChain is an open-source, Rust-based framework designed to simplify the development of blockchain-agnostic data pipelines. It provides a set of libraries and tools for building scalable, decentralized data processing applications.

Turn 2
User: What problem does it solve?
AI:   LangChain solves the problem of building blockchain-agnostic data pipelines by abstracting away the complexity of blockchain-specific interactions, allowing developers to focus on building data processing applications that can work across multiple blockchain platforms.

Turn 3
User: What was my first question?
AI:   Your first question was "What is LangChain?"

Total messages in history: 7
The model remembers context because we pass the full history each time!


### Experiment 5B: Conversation With a Window (Last K Turns)

In production, you can't grow the message list forever. A simple solution: keep only the last K exchanges.

In [22]:
def windowed_chat(messages_history, user_input, system_prompt, window_size=2):
    """Chat with a sliding window — keep only last K exchanges."""
    messages_history.append(HumanMessage(content=user_input))

    # Build the windowed context: system + last K pairs
    windowed = [SystemMessage(content=system_prompt)]
    # Keep last window_size * 2 messages (each exchange = human + ai)
    windowed.extend(messages_history[-(window_size * 2 + 1):])

    response = llm.invoke(windowed)
    messages_history.append(response)
    return response.content


# Test it — the model should "forget" early turns
history = []
system = "You are a helpful assistant. Keep answers to 1 sentence."

questions = [
    "My favorite color is blue. Remember that.",
    "What is 2 + 2?",
    "What is the capital of France?",
    "What is my favorite color?"  # Window=2 means this is likely forgotten
]

for i, q in enumerate(questions, 1):
    print(f"\nTurn {i}: {q}")
    print(f"History:{history}")
    answer = windowed_chat(history, q, system, window_size=2)
    print(f"AI:     {answer}")

print("\nWith window_size=2, the model only sees the last 2 exchanges.")
print("Earlier context (like favorite color) may be forgotten!")


Turn 1: My favorite color is blue. Remember that.
History:[]
AI:     I've taken note that your favorite color is blue.

Turn 2: What is 2 + 2?
History:[HumanMessage(content='My favorite color is blue. Remember that.', additional_kwargs={}, response_metadata={}), AIMessage(content="I've taken note that your favorite color is blue.", additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-03-08T12:19:11.132218Z', 'done': True, 'done_reason': 'stop', 'total_duration': 840163125, 'load_duration': 124305209, 'prompt_eval_count': 47, 'prompt_eval_duration': 541009208, 'eval_count': 12, 'eval_duration': 170078379, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019ccd63-1b4f-7ce1-9fe4-ab81dca12b1f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 12, 'total_tokens': 59})]
AI:     The answer is 4.

Turn 3: What is the capital of France?
History:[HumanMessage(content='My favorite c

---

## 6. Pillar 3 — Swappability: The Power of Abstractions

LangChain's biggest value: **all providers implement the same interface**.

Switching from OpenAI to Ollama (or Claude, Gemini, etc.) requires changing **one line** — your chain logic stays the same.

```python
# Switch providers — chain logic is IDENTICAL
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_anthropic import ChatAnthropic

llm = ChatOpenAI(model="gpt-4o-mini")       # OpenAI
llm = ChatOllama(model="llama3.2:3b")        # Ollama (local)
llm = ChatAnthropic(model="claude-3-sonnet")  # Anthropic

# Same chain works with ANY of the above
chain = prompt | llm | parser
```

### Experiment 6A: Same Chain, Different Model Configs

In [23]:
# Demonstrate swappability with different Ollama configurations
# (Same concept applies when swapping between OpenAI, Anthropic, etc.)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Explain what a 'chain' is in LangChain. One sentence.")
])
parser = StrOutputParser()

# Config 1: Creative (high temperature)
creative_llm = ChatOllama(model=MODEL, temperature=1.0)
creative_chain = prompt | creative_llm | parser

# Config 2: Precise (low temperature)
precise_llm = ChatOllama(model=MODEL, temperature=0.0)
precise_chain = prompt | precise_llm | parser

print("SAME chain structure, DIFFERENT model configs:\n")

print("Creative (temp=1.0):")
print(creative_chain.invoke({}))

print("\nPrecise (temp=0.0):")
print(precise_chain.invoke({}))

print("\nThe chain logic (prompt | llm | parser) is identical.")
print("Only the model configuration changed!")

SAME chain structure, DIFFERENT model configs:

Creative (temp=1.0):
In LangChain, a chain refers to a graph-like data structure that represents a collection of nodes (key-value pairs, tasks, or other entities) connected by edges (dependencies or relationships), allowing for efficient querying, updating, and traversing of the data.

Precise (temp=0.0):
In LangChain, a chain refers to a data structure that represents a sequence of linked data objects, such as files, folders, or other resources, that can be traversed and manipulated in a hierarchical manner.

The chain logic (prompt | llm | parser) is identical.
Only the model configuration changed!


---

## 7. Output Parsers — Structured Output from LLMs

LLMs return raw text. **Output parsers** convert that text into structured data.

| Parser | Output |
|--------|---------|
| `StrOutputParser` | Plain string (strips the AIMessage wrapper) |
| `JsonOutputParser` | Parsed JSON dict |
| `CommaSeparatedListOutputParser` | Python list from comma-separated text |

### Experiment 7A: StrOutputParser vs Raw Response

In [24]:
# Without parser — you get an AIMessage object
raw_chain = prompt | llm
raw_result = raw_chain.invoke({})
print(f"Without parser: {type(raw_result).__name__}")
print(f"  Content: {raw_result.content}")

print()

# With parser — you get a clean string
parsed_chain = prompt | llm | StrOutputParser()
parsed_result = parsed_chain.invoke({})
print(f"With StrOutputParser: {type(parsed_result).__name__}")
print(f"  Content: {parsed_result}")

Without parser: AIMessage
  Content: In LangChain, a chain refers to a collection of linked data structures, such as nodes, edges, or blocks, that represent a graph or a sequence of data, often used to model complex relationships between entities.

With StrOutputParser: TextAccessor
  Content: In LangChain, a 'chain' refers to a hierarchical structure of interconnected nodes that represent a sequence of tasks, assets, or operations, allowing for modular and reusable code organization.


### Experiment 7B: Comma-Separated List Parser

In [25]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

list_parser = CommaSeparatedListOutputParser()

list_prompt = ChatPromptTemplate.from_messages([
    ("system", "You list items separated by commas. No numbering, no extra text."),
    ("human", "List 5 {category}.")
])

list_chain = list_prompt | llm | list_parser

result = list_chain.invoke({"category": "programming languages"})

print(f"Type: {type(result).__name__}")  # list!
print(f"Result: {result}")
print(f"First item: {result[0]}")
print(f"Count: {len(result)}")

Type: list
Result: ['Java', 'Python', 'JavaScript', 'C++', 'Ruby']
First item: Java
Count: 5


### Experiment 7C: JSON Output Parser

In [26]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

json_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a data assistant. Always respond with valid JSON only. "
     "No markdown, no explanation — just the JSON object."),
    ("human",
     "Give me info about {topic}. "
     'Return JSON with keys: "name", "category", "description" (1 sentence).')
])

json_chain = json_prompt | llm | json_parser

result = json_chain.invoke({"topic": "LangChain"})

print(f"Type: {type(result).__name__}")  # dict!
print(f"Result: {result}")

if isinstance(result, dict):
    for key, value in result.items():
        print(f"  {key}: {value}")

Type: dict
Result: {'name': 'LangChain', 'category': 'AI Framework', 'description': 'LangChain is an open-source framework for building conversational AI applications.'}
  name: LangChain
  category: AI Framework
  description: LangChain is an open-source framework for building conversational AI applications.


---

## 8. Pillar 4 & 5 — LangGraph & LangSmith (Overview)

### LangGraph — Agents with Graph-Based Control Flow

Chains are **linear**: Step A → Step B → Step C.

But real agents need **cycles**: reason → act → observe → decide to continue or try a different approach.

**LangGraph** provides:
- **State machines** — persistent state across steps
- **Nodes** — individual processing steps (LLM calls, tools, human input)
- **Edges** — connections with conditional routing
- **Human-in-the-loop** — pause for human approval

| Feature | LangChain Chains | LangGraph |
|---------|------------------|-----------|
| Flow | Linear | Cyclical, conditional |
| State | Stateless | Stateful + persistent |
| Control | Implicit | Explicit graph |
| Use case | Simple pipelines | Complex agents |

### LangSmith — Observability & Monitoring

When an LLM app gives a wrong answer — was it the prompt? The retrieved context? The model? LangSmith provides:

- **Tracing** — see every step of every chain execution
- **Evaluation** — automated testing of LLM outputs
- **Monitoring** — track latency, cost, errors in production
- **Datasets** — curate test cases for regression testing

> LangSmith integrates automatically — just set an environment variable and every execution is traced.

---

## 9. Putting It All Together — A Multi-Step Chain

Let's build a practical example that combines prompt templates, LCEL chains, and output parsing.

### Experiment 9A: Two-Step Chain — Generate then Analyze

In [27]:
# Step 1: Generate a concept explanation
explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer. Explain concepts clearly in 2-3 sentences."),
    ("human", "Explain: {concept}")
])

explain_chain = explain_prompt | llm | StrOutputParser()

# Step 2: Simplify the explanation for a child
simplify_prompt = ChatPromptTemplate.from_messages([
    ("system", "You simplify technical explanations for a 10-year-old. Use an analogy. 2 sentences max."),
    ("human", "Simplify this: {explanation}")
])

simplify_chain = simplify_prompt | llm | StrOutputParser()

# Run them in sequence
concept = "vector embeddings"

print(f"Concept: {concept}")
print(f"\n{'=' * 50}")
print("Step 1 — Technical Explanation:")
print(f"{'=' * 50}")

explanation = explain_chain.invoke({"concept": concept})
display(Markdown(explanation))

print(f"\n{'=' * 50}")
print("Step 2 — Simplified for a 10-year-old:")
print(f"{'=' * 50}")

simple = simplify_chain.invoke({"explanation": explanation})
display(Markdown(simple))

Concept: vector embeddings

Step 1 — Technical Explanation:


**Vector Embeddings:**

Vector embeddings are a way to represent complex, high-dimensional data (such as text, images, or audio) as compact, low-dimensional vectors. This process, called dimensionality reduction, captures the essential features and relationships of the data in a more manageable form, allowing for efficient storage, computation, and comparison. By transforming data into vectors, we can apply various machine learning algorithms, such as neural networks, to extract insights and make predictions.


Step 2 — Simplified for a 10-year-old:


Think of vector embeddings like a super-compact address book. Instead of writing down all the details about a person, you just write down their name and a few important characteristics, like where they live and what they like, so you can quickly find them.

### Experiment 9B: Chaining with RunnableLambda (Custom Logic in a Chain)

In [28]:
from langchain_core.runnables import RunnableLambda

# Custom function wrapped as a Runnable
def format_as_flashcard(text):
    """Format LLM output as a study flashcard."""
    lines = text.strip().split("\n")
    title = lines[0] if lines else "Concept"
    body = "\n".join(lines[1:]) if len(lines) > 1 else text
    return f"{'=' * 40}\n  FLASHCARD\n{'=' * 40}\n{text}\n{'=' * 40}"

# Build the chain: prompt → model → parse → custom format
flashcard_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Give a brief definition (1-2 sentences) then one key example."),
        ("human", "Define: {term}")
    ])
    | llm
    | StrOutputParser()
    | RunnableLambda(format_as_flashcard)  # Custom post-processing
)

terms = ["LCEL", "Prompt Template", "Output Parser"]

for term in terms:
    result = flashcard_chain.invoke({"term": term})
    print(result)
    print()

  FLASHCARD
LCEL stands for "London Colney Elevated Railway", a proposed high-speed rail line in the United Kingdom.

Key Example: The line was initially proposed in 1937 as a way to improve transportation links between London and the surrounding areas, with the aim of reducing congestion on existing rail lines.

  FLASHCARD
**Definition:** A prompt template is a pre-designed structure or framework used to guide the creation of text-based responses, such as articles, social media posts, or chatbot conversations. It provides a standardized format and set of questions to help generate coherent and relevant content.

**Example:** A company might use a prompt template for customer support chatbots, which includes a standard set of questions such as "What is your issue?", "Can you please provide more details?", and "What is your desired resolution?". This template ensures that the chatbot's responses are consistent, helpful, and follow a specific format.

  FLASHCARD
An Output Parser is a s

---

## 10. Package Installation Strategy

Install **only what you need**. The modular structure prevents dependency bloat.

| Use Case | Packages |
|----------|----------|
| Basic LLM apps (local) | `langchain langchain-core langchain-ollama` |
| Basic LLM apps (OpenAI) | `langchain langchain-core langchain-openai` |
| RAG applications | `langchain langchain-ollama langchain-chroma langchain-community` |
| Agent applications | `langgraph langchain-ollama` |
| Observability | `langsmith` |

```bash
# What we installed for this notebook:
pip install langchain langchain-core langchain-ollama
```

---

## 11. Sandbox — Try It Yourself!

In [29]:
# ============================================================
#  SANDBOX - Edit and re-run!
# ============================================================

# Build your own chain
my_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep it brief."),
    ("human", "{question}")
])

my_chain = my_prompt | llm | StrOutputParser()

# Try different questions
question = "What makes LangChain different from calling an LLM API directly?"

use_streaming = True  # Toggle: True for streaming, False for invoke

# ============================================================

if use_streaming:
    print("[STREAMING]\n")
    for chunk in my_chain.stream({"question": question}):
        print(chunk, end="", flush=True)
    print()
else:
    print("[INVOKE]\n")
    result = my_chain.invoke({"question": question})
    display(Markdown(result))

[STREAMING]

LangChain is different from calling an LLM API directly in several ways:

1. **Decoupling**: LangChain allows for decoupling the LLM from the application logic, making it easier to switch between different LLM models or integrate with other tools.
2. **Modular architecture**: LangChain provides a modular architecture, enabling developers to focus on specific tasks or workflows without having to manage the underlying LLM infrastructure.
3. **Workflow-based interactions**: LangChain enables developers to define workflows that interact with the LLM, rather than making direct API calls. This provides a more intuitive and user-friendly interface.
4. **Flexibility**: LangChain allows for flexible integration with various LLM models, frameworks, and tools, making it easier to adapt to changing requirements.

By decoupling the LLM from the application logic, LangChain provides a more scalable, maintainable, and flexible solution for integrating LLMs into applications.


---

## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **LangChain** | An ecosystem of 5 pillars — Core, LangChain, Community, LangGraph, LangSmith |
| **Core Abstractions** | Everything implements the Runnable protocol: `invoke()`, `stream()`, `batch()` |
| **Message Types** | `SystemMessage`, `HumanMessage`, `AIMessage` — structured, not raw dicts |
| **Prompt Templates** | Reusable, parameterized prompts with `ChatPromptTemplate` |
| **LCEL** | Compose chains with `\|` pipe operator: `prompt \| model \| parser` |
| **Output Parsers** | Convert raw LLM text to strings, lists, JSON, or custom formats |
| **Memory** | LLMs have no memory — you manage conversation history yourself |
| **Swappability** | Same chain works with any provider — change one import line |
| **Modular Install** | Install only what you need: `langchain-ollama`, `langchain-openai`, etc. |
| **LangGraph** | For stateful agents needing cycles, conditions, and persistence |
| **LangSmith** | Observability — tracing, evaluation, and monitoring for production |